# Methods-Guided Reanalysis of GSE150903

This notebook is a redo of the first exploratory analysis using the paper's supplementary methods as the guide. The main change is that the GEO matrix is treated as a processed SCTransform-derived matrix, not as raw UMI counts.

## Python Reproduction Strategy

The original paper analyzed the data in Seurat/R. This notebook reproduces the downstream analysis in Python with Scanpy, using the processed SCT-scaled matrix released on GEO.

| Paper method or parameter | Python/Scanpy implementation in this notebook |
| --- | --- |
| CellRanger Count v3.1.0 with STAR/GRCh38 | documented as the upstream source of the GEO matrix; not rerun here |
| Seurat v3 object | `scanpy.AnnData` object |
| merged sample matrices | load GEO matrix, transpose to cells x genes, add sample labels |
| mitochondrial percentage >30% removed | documented; exact re-filtering requires raw counts |
| likely doublets removed using `nCount_RNA` | documented as equivalent to `total_counts`; exact re-filtering requires raw counts |
| final dataset of 32,464 cells | checked against the processed GEO matrix dimensions |
| SCTransform normalization/scaling/variable features | use the released SCT-scaled matrix directly; do not re-normalize as raw counts |
| regress mitochondrial percentage and cell cycle | documented as already part of the paper's Seurat workflow; exact rerun requires raw counts/Seurat |
| ElbowPlot selected 4 PCs | use `sc.pp.neighbors(..., n_pcs=4)` for main clustering |
| Seurat `FindNeighbors` | `sc.pp.neighbors` |
| Seurat `FindClusters` | `sc.tl.leiden` |
| UMAP | `sc.tl.umap` |
| top 10 differentially expressed genes | `sc.tl.rank_genes_groups(..., method="wilcoxon")` |
| known marker gene interpretation | Scanpy dotplots, UMAP feature plots, and marker-set scoring |
| mature ChP subclustering with PCs 1-12 | optional mature/ChP-like subset with `n_pcs=12` |

So the notebook is not switching to R. It is a Python translation of the paper's reported Seurat workflow, with clear notes where exact reproduction would require raw counts or the original Seurat pipeline.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy import sparse

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=120, frameon=False)

PROJECT_DIR = Path.cwd()
DATA_PATH = PROJECT_DIR / "data" / "processed" / "GSE150903_SCT_scaled_count_matrix.txt"
FIGURE_DIR = PROJECT_DIR / "results" / "figures"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## Seurat Metadata Crosswalk

The paper mentions Seurat metadata columns that have close Scanpy equivalents when raw counts are available:

| Seurat/R term | Meaning | Scanpy/Python equivalent |
| --- | --- | --- |
| `nCount_RNA` | total UMI/RNA counts per cell | `adata.obs["total_counts"]` |
| `nFeature_RNA` | number of detected genes per cell | `adata.obs["n_genes_by_counts"]` |
| mitochondrial percentage | percent mitochondrial counts per cell | `adata.obs["pct_counts_mt"]` |
| cell cycle scores | S-phase/G2M marker-gene scores | `sc.tl.score_genes_cell_cycle` |
| Seurat clusters | active cluster identity | `adata.obs["leiden"]` or another annotation column |

Important limitation: these QC columns can be reproduced exactly only from raw counts. The GEO matrix used here is already SCTransform-derived, so this notebook focuses on the downstream reproduction: PCA, neighbors, clustering, UMAP, marker genes, annotation, and final figures.

## 1. Load GEO Processed Matrix

The GEO supplementary file is `GSE150903_SCT_scaled_count_matrix.txt.gz`. After unzipping, place it at `data/processed/GSE150903_SCT_scaled_count_matrix.txt`. Rows are genes and columns are cells, so the matrix is transposed for AnnData.

In [ ]:
counts = pd.read_csv(DATA_PATH, sep="\t", index_col=0)
print("Original matrix shape, genes x cells:", counts.shape)

adata = sc.AnnData(counts.T)
adata.var_names_make_unique()

print("AnnData shape, cells x genes:", adata.shape)
adata

## 2. Add Sample Metadata

The cell barcode prefixes encode the sample identity. These labels follow the paper/GEO sample descriptions.

In [ ]:
adata.obs["prefix"] = adata.obs_names.str.split("_").str[0]

sample_map = {
    "T": "Telencephalon organoids D55",
    "1": "Choroid Plexus Org D27",
    "2": "Choroid Plexus Org D46",
    "3": "Choroid Plexus Org D53",
}

adata.obs["sample"] = adata.obs["prefix"].map(sample_map).astype("category")

display(adata.obs["sample"].value_counts())
print("Expected final high-quality cells from paper: 32464")
print("Observed cells in processed matrix:", adata.n_obs)

## 3. Main PCA, Neighbors, Clustering, and UMAP

The paper used ElbowPlot and selected 4 principal components for the main analysis. Here, Scanpy is used as the Python analogue. Leiden clustering is not identical to Seurat `FindClusters`, so cluster labels may not match one-for-one.

In [ ]:
sc.tl.pca(adata, svd_solver="arpack")
sc.pl.pca_variance_ratio(adata, log=True, n_pcs=30)

sc.pp.neighbors(adata, n_neighbors=15, n_pcs=4)
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.5, flavor="igraph", n_iterations=2, directed=False)

sc.pl.umap(adata, color=["sample", "leiden"], wspace=0.4)

## 4. Cluster Composition By Sample

In [ ]:
cluster_sample_counts = pd.crosstab(adata.obs["leiden"], adata.obs["sample"])
cluster_sample_percent = pd.crosstab(adata.obs["leiden"], adata.obs["sample"], normalize="index") * 100

display(cluster_sample_counts)
display(cluster_sample_percent.round(2))

## 5. Marker Genes Per Cluster

The paper used the top 10 differentially expressed genes and known markers to identify clusters.

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="leiden", method="wilcoxon")
sc.pl.rank_genes_groups(adata, n_genes=10, sharey=False)

marker_table = sc.get.rank_genes_groups_df(adata, group=None)
marker_table.to_csv(PROCESSED_DIR / "methods_guided_leiden_marker_genes.csv", index=False)
display(marker_table.head(20))

## 6. Marker Sets From Choroid Plexus Biology

These marker sets combine genes reported in the paper with common cell-state and tissue markers used for interpretation.

In [ ]:
marker_sets = {
    "Choroid plexus": ["CLIC6", "HTR2C", "TTR", "AQP1"],
    "Immature ChP / hem": ["MSX1", "OTX2", "RSPO3", "PAX6"],
    "Mature ChP epithelium": ["TTR", "KRT18", "NME5"],
    "ChP stroma / mesenchymal": ["COL1A1", "LUM", "DCN", "DLK1"],
    "Barrier / tight junction": ["CLDN1", "CLDN3", "CLDN5", "TJP1", "TJP2", "OCLN"],
    "CSF secretion / transport": ["AQP1", "CA2", "CA12", "SLC23A2", "SLC46A1"],
    "Telencephalon / neuronal": ["DCX", "FOXG1", "GAP43"],
    "Light / ciliated ChP": ["FOXJ1", "ARL13B", "CCDC67"],
    "Dark / mitochondria-rich ChP": ["CARD19", "IGF2", "RBP1"],
    "Myoepithelial-like ChP": ["KRT17", "ACTA2", "TAGLN"],
    "Dividing / cycling": ["MKI67", "TOP2A", "PCNA"],
}

marker_sets_present = {
    group: [gene for gene in genes if gene in adata.var_names]
    for group, genes in marker_sets.items()
}
marker_sets_present = {group: genes for group, genes in marker_sets_present.items() if genes}

for group, genes in marker_sets_present.items():
    print(f"{group}: {genes}")

sc.pl.dotplot(adata, var_names=marker_sets_present, groupby="leiden", standard_scale="var", dendrogram=False)

## 7. Rule-Assisted Cluster Annotation

This section scores cells for each marker group, averages scores by Leiden cluster, and proposes a first-pass annotation. Manual review is still required.

In [ ]:
import re

score_columns = {}
for cell_type, genes in marker_sets_present.items():
    safe_name = re.sub(r"[^A-Za-z0-9_]+", "_", cell_type).strip("_")
    score_col = f"score_{safe_name}"
    score_columns[cell_type] = score_col
    sc.tl.score_genes(adata, gene_list=genes, score_name=score_col)

cluster_scores = adata.obs.groupby("leiden", observed=True)[list(score_columns.values())].mean()
cluster_scores = cluster_scores.rename(columns={v: k for k, v in score_columns.items()})

auto_labels = cluster_scores.idxmax(axis=1).rename("auto_cell_type")
annotation_table = cluster_scores.copy()
annotation_table.insert(0, "auto_cell_type", auto_labels)
annotation_table.insert(1, "reviewed_cell_type", auto_labels)

annotation_table.to_csv(PROCESSED_DIR / "methods_guided_auto_cluster_annotation.csv")
display(annotation_table)

cluster_to_label = auto_labels.to_dict()
adata.obs["cell_type_auto"] = adata.obs["leiden"].map(cluster_to_label).astype("category")
sc.pl.umap(adata, color=["sample", "leiden", "cell_type_auto"], wspace=0.4)

## 8. Final Summary Figures

After reviewing the annotation table, update `cell_type_reviewed` if needed. For now, this template uses the automatic label as the reviewed label.

In [ ]:
adata.obs["cell_type_reviewed"] = adata.obs["cell_type_auto"].copy()

cell_type_percent = pd.crosstab(adata.obs["sample"], adata.obs["cell_type_reviewed"], normalize="index") * 100
ax = cell_type_percent.plot(kind="bar", stacked=True, figsize=(12, 6), width=0.8)
ax.set_ylabel("Percent of cells")
ax.set_xlabel("Sample")
ax.set_title("Methods-Guided Cell-Type Composition By Sample")
ax.legend(title="Reviewed cell type", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "methods_guided_cell_type_composition_by_sample.png", bbox_inches="tight", dpi=300)
plt.show()

sc.pl.dotplot(
    adata,
    var_names=marker_sets_present,
    groupby="cell_type_reviewed",
    standard_scale="var",
    dendrogram=False,
    save=False,
)

## 9. Optional Mature ChP Subclustering

The paper subclustered the mature ChP cluster using PCs 1-12. Once mature ChP-like cells are identified, subset them here and rerun PCA/neighbors/Leiden/UMAP with `n_pcs=12`.

In [ ]:
mature_mask = adata.obs["cell_type_reviewed"].astype(str).str.contains("Mature ChP|Choroid plexus", case=False, regex=True)
adata_mature = adata[mature_mask].copy()

print("Mature/ChP-like cells selected:", adata_mature.n_obs)

if adata_mature.n_obs > 100:
    sc.tl.pca(adata_mature, svd_solver="arpack")
    sc.pp.neighbors(adata_mature, n_neighbors=15, n_pcs=12)
    sc.tl.umap(adata_mature)
    sc.tl.leiden(adata_mature, resolution=0.5, flavor="igraph", n_iterations=2, directed=False)
    sc.pl.umap(adata_mature, color=["sample", "leiden"], wspace=0.4)
else:
    print("Not enough mature/ChP-like cells selected for subclustering; review labels first.")

## 10. Save Methods-Guided AnnData

In [ ]:
adata.write_h5ad(PROCESSED_DIR / "methods_guided_reanalysis.h5ad")
print("Saved:", PROCESSED_DIR / "methods_guided_reanalysis.h5ad")